# HCC Survival Prediction - Example Usage

This notebook demonstrates how to use the modular code in the `src/` directory.

## Setup

In [ ]:
# Add src to path
import sys
sys.path.append('../src')

# Import our modules
from data_preprocessing import preprocess_pipeline
from model_training import train_all_models, save_all_models
from evaluation import evaluate_all_models, plot_model_comparison

# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Data Preprocessing

Load and preprocess the HCC dataset using our preprocessing pipeline.

In [ ]:
# Run preprocessing pipeline
data = preprocess_pipeline(
    filepath='../data/raw/hcc_dataset.csv',
    missing_strategy='median',
    test_size=0.2,
    random_state=RANDOM_STATE
)

# Extract preprocessed data
X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']
feature_names = data['feature_names']
target_encoder = data['target_encoder']

print(f"\nClass labels: {dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))}")

## 2. Model Training

Train all models (Random Forest, Decision Tree, KNN).

In [ ]:
# Train all models
models = train_all_models(
    X_train,
    y_train,
    hyperparameter_tuning=False,  # Set to True for grid search
    random_state=RANDOM_STATE
)

## 3. Model Evaluation

Evaluate all models and compare their performance.

In [ ]:
# Evaluate all models
results_df = evaluate_all_models(
    models,
    X_test,
    y_test,
    class_names=target_encoder.classes_.tolist(),
    save_dir='../results'
)

In [ ]:
# Plot model comparison
plot_model_comparison(
    results_df,
    save_path='../results/figures/model_comparison.png'
)

## 4. Save Models

Save trained models for future use.

In [ ]:
# Save all models
save_all_models(models, output_dir='../models')

## 5. Feature Importance (Random Forest)

Analyze which features are most important for prediction.

In [ ]:
# Get feature importance from Random Forest
rf_model = models['Random Forest']
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

# Plot top 15 features
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance')
plt.title('Top 15 Most Important Features (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../results/figures/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))

## Summary

This notebook demonstrated:
1. ✅ Loading and preprocessing data using modular functions
2. ✅ Training multiple models with consistent interface
3. ✅ Comprehensive model evaluation and comparison
4. ✅ Saving models for deployment
5. ✅ Feature importance analysis

All code is now modular, reusable, and well-documented!